# 01 · Build the action-recognition eval set (LSO-68)

Extracts **person crops** from the office stills in `notebooks/input/` plus one
random frame each from `cam1.mp4` / `cam2.mp4`, then writes them to a folder for
you to annotate by hand.

The crops are cut the same way production cuts them — a **tight** bbox crop,
`frame[y1:y2, x1:x2]`, matching `camera_engine.py:410-412`, which is the exact
`proof_image` the running service hands to `ActionRecognizer`. Deliberately *not*
`crop_person_roi(expand=0.1)`: that is the face path, and LSO-68 scopes
preprocessing changes out.

**Output tree**

```
notebooks/eval/action/
  crops/            <- annotate these
  contact_sheets/   <- numbered grids, for a fast visual pass
  manifest.json     <- crop_id -> source, frame_num, bbox, conf, w, h
```

Re-running is safe: crops already on disk are skipped, and crops you have
already renamed or moved into a class folder are left alone.

In [ ]:
import hashlib
import json
import sys
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("ignore")

from lum_vision import ModelFactory, VisionConfig

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EVAL = REPO / "notebooks" / "eval" / "action"
CROPS = EVAL / "crops"
SHEETS = EVAL / "contact_sheets"
for d in (CROPS, SHEETS):
    d.mkdir(parents=True, exist_ok=True)

print("repo :", REPO)
print("eval :", EVAL)
print("python:", sys.version.split()[0])

## 1 · Configuration

`MIN_*` thresholds mirror `crop_quality` in `configs/global_tracking.yaml`
(the gate ReID already applies), so the eval set only contains crops production
would actually accept.

In [ ]:
# --- sources -------------------------------------------------------------
IMAGE_DIR = REPO / "notebooks" / "input"
# 00_input.png is a 5.8 MB raw frame grab and sandisk-*.jpg is stock, not office
# footage - the perf notebook excludes both and so do we.
EXCLUDE = {"00_input.png", "sandisk-pAKgXLu04CQ-unsplash.jpg"}
IMAGE_SOURCES = sorted(p for p in IMAGE_DIR.glob("*.*") if p.name not in EXCLUDE)
VIDEO_SOURCES = [p for p in (REPO / "cam1.mp4", REPO / "cam2.mp4") if p.exists()]

# --- sampling ------------------------------------------------------------
# Deliberately small: every crop here is a crop somebody has to label by hand.
# The stills carry the set; the videos contribute one frame each for a bit of
# scene variety.
SAMPLES_PER_VIDEO = 1       # randomly chosen frames per video (seeded)
RANDOM_SEED = 42            # so the same frames come back on a re-run
MAX_CROPS_PER_FRAME = 8     # with only ~8 frames in total there is no crowded-
                            # frame problem to guard against, so keep everyone
                            # visible rather than discarding usable people
MAX_CROPS_PER_SOURCE = 60   # safety cap, keeps one video from dominating
MIN_CONF = 0.5              # YOLO person confidence floor

# --- crop quality gate (configs/global_tracking.yaml -> crop_quality) -----
MIN_HEIGHT = 80
MIN_WIDTH = 40
MIN_AREA = 3200

print(f"{len(IMAGE_SOURCES)} images, {len(VIDEO_SOURCES)} videos")
for p in IMAGE_SOURCES + VIDEO_SOURCES:
    print("  ", p.name)

## 2 · Person detector

`PersonDetector` exposes no `detect()` wrapper, so we call the underlying
ultralytics model and filter class 0 ourselves — the same pattern
`lum_vision_demo.ipynb` uses.

In [ ]:
config = VisionConfig()  # default model_cache_dir (~/.cache/lum-vision).
                         # Do NOT point this at ./volumes/models - it is root-owned.
models = ModelFactory(config)

_ = models.person_detector  # force the lazy load now, not mid-loop
print("person model :", config.person_detection_model, config.person_model_size)
print("conf threshold:", max(MIN_CONF, config.person_detection_threshold))

In [ ]:
def detect_persons(frame: np.ndarray) -> list[dict]:
    """Return [{'bbox': [x1,y1,x2,y2], 'confidence': float}] for class-0 boxes."""
    threshold = max(MIN_CONF, config.person_detection_threshold)
    result = models.person_detector.model(frame, verbose=False)[0]
    out = []
    for box in result.boxes:
        if int(box.cls) != 0:
            continue
        conf = float(box.conf)
        if conf < threshold:
            continue
        x1, y1, x2, y2 = (int(v) for v in box.xyxy[0])
        out.append({"bbox": [x1, y1, x2, y2], "confidence": conf})
    return out

## 3 · Frame iterator

Images yield their single frame; each video contributes `SAMPLES_PER_VIDEO`
randomly-chosen frames, seeded so a re-run picks the same ones and any crops
you have already labelled stay valid.

The set is kept small on purpose — every crop is one a human has to label. Raise
`SAMPLES_PER_VIDEO` if a class turns out too thin to score.

In [ ]:
def frame_iter(is_done=lambda source_id: False):
    """Yield (source_id, frame_num, frame) across every configured source.

    ``is_done`` is polled before each seek so a video that has already hit
    MAX_CROPS_PER_SOURCE is abandoned instead of being decoded to the end -
    cam2.mp4 alone is 200 MB.
    """
    for path in IMAGE_SOURCES:
        frame = cv2.imread(str(path))
        if frame is None:
            print(f"  ! unreadable, skipped: {path.name}")
            continue
        yield path.stem[:24], 0, frame

    for path in VIDEO_SOURCES:
        cap = cv2.VideoCapture(str(path))
        if not cap.isOpened():
            print(f"  ! could not open: {path.name}")
            continue
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        # Random frames, drawn from the middle 90% so we never land on a black
        # lead-in or tail. Seeded, so a re-run picks the same frames and the
        # crops you already labelled stay valid.
        # Seeded per video, not once: a shared seed would hand both cameras the
        # same frame index, and these are simultaneous recordings of one office.
        rng = np.random.default_rng(RANDOM_SEED + sum(path.stem.encode()))
        positions = sorted(rng.integers(int(total * 0.05), int(total * 0.95), SAMPLES_PER_VIDEO))
        print(f"  {path.name}: {total} frames @ {fps:.1f}fps ({total / fps / 60:.1f} min) "
              f"-> frame(s) {[int(p) for p in positions]}")
        try:
            for pos in positions:
                if is_done(path.stem):
                    print(f"  {path.name}: crop cap reached, moving on")
                    break
                frame_num = int(pos)
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
                ok, frame = cap.read()
                if not ok:
                    continue
                yield path.stem, frame_num, frame
        finally:
            cap.release()

## 4 · Extract crops

Tight bbox crop, clamped to frame bounds — the same geometry as
`camera_engine._clip_bbox` + `frame[y1:y2, x1:x2].copy()`.

Filenames carry the label: `{label}__{source}_{frame:06d}_p{idx}_{hash8}.jpg`.
Fresh crops ship as `unlabeled__...`; the `hash8` of the source frame + bbox keeps
re-runs idempotent.

In [ ]:
def clip_bbox(frame, bbox):
    """Clamp a bbox to frame bounds. Mirrors camera_engine._clip_bbox."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = (int(v) for v in bbox)
    return max(0, x1), max(0, y1), min(w, x2), min(h, y2)


def bbox_area(det) -> int:
    x1, y1, x2, y2 = det["bbox"]
    return max(0, x2 - x1) * max(0, y2 - y1)


def crop_is_usable(crop) -> bool:
    h, w = crop.shape[:2]
    return h >= MIN_HEIGHT and w >= MIN_WIDTH and h * w >= MIN_AREA


def existing_crop_ids() -> set[str]:
    """Crop ids already on disk, wherever they were renamed or moved to.

    Recursive so that crops you have already labelled - by renaming the prefix
    or by dragging them into a class subfolder - are never re-extracted.
    """
    return {p.stem.split("__", 1)[1] for p in CROPS.rglob("*.jpg") if "__" in p.stem}

In [ ]:
known = existing_crop_ids()
print(f"{len(known)} crop(s) already on disk - they will be skipped\n")

manifest_path = EVAL / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

per_source: dict[str, int] = {}
new_crops = 0
frames_seen = 0

def source_is_full(source_id: str) -> bool:
    return per_source.get(source_id, 0) >= MAX_CROPS_PER_SOURCE


for source_id, frame_num, frame in frame_iter(source_is_full):
    frames_seen += 1
    if source_is_full(source_id):
        continue

    # Largest boxes first, then keep only the top few: a distant 45px figure
    # tells a VLM nothing, and one crowded frame should not swamp the set.
    dets = sorted(detect_persons(frame), key=bbox_area, reverse=True)[:MAX_CROPS_PER_FRAME]

    for idx, det in enumerate(dets):
        if source_is_full(source_id):
            break

        x1, y1, x2, y2 = clip_bbox(frame, det["bbox"])
        if x2 <= x1 or y2 <= y1:
            continue
        crop = frame[y1:y2, x1:x2].copy()   # tight, exactly like production
        if not crop_is_usable(crop):
            continue

        digest = hashlib.sha1(f"{source_id}|{frame_num}|{x1},{y1},{x2},{y2}".encode()).hexdigest()[:8]
        crop_id = f"{source_id}_{frame_num:06d}_p{idx}_{digest}"
        per_source[source_id] = per_source.get(source_id, 0) + 1
        if crop_id in known:
            continue

        cv2.imwrite(str(CROPS / f"unlabeled__{crop_id}.jpg"), crop)
        manifest[crop_id] = {
            "source": source_id,
            "frame_num": frame_num,
            "bbox": [x1, y1, x2, y2],
            "confidence": round(det["confidence"], 4),
            "width": x2 - x1,
            "height": y2 - y1,
        }
        known.add(crop_id)
        new_crops += 1

    if frames_seen % 25 == 0:
        print(f"  {frames_seen} frames scanned, {new_crops} new crops")

manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"\nscanned {frames_seen} frames -> {new_crops} new crops ({len(manifest)} total)")

## 5 · Contact sheets

Numbered grids so you can eyeball the whole set before renaming anything.

This is a viewing aid only - crops are letterboxed into their cell (padded, not
stretched) with a gutter between cells so neighbouring people don't run
together. Nothing here touches the saved crop files themselves: those stay the
exact tight production crop that notebook 2 feeds to the model.

In [ ]:
CELL_W, CELL_H = 130, 240
GUTTER = 8
BG = (32, 32, 32)
COLS, ROWS = 10, 5
PER_SHEET = COLS * ROWS
# Guarantees >= 20% margin on *every* side of every crop: without this, a crop
# whose aspect ratio happens to match the cell's gets 0% margin on one axis
# while another gets 55%+ on the other - purely a function of shape, not a
# deliberate choice. Capping content at 60% of the cell (20% margin per side,
# both sides) removes that variance.
PADDING_PER_SIDE = 0.20


def fit_letterbox(img, w, h, padding_per_side=PADDING_PER_SIDE):
    """Resize preserving aspect ratio, padded (not stretched) onto a w x h canvas.

    `padding_per_side` caps the fitted image at (1 - 2*padding_per_side) of
    each cell dimension, so every crop gets at least that much margin on
    *every* side - not just whichever axis the aspect-ratio fit happens to
    leave short.
    """
    ih, iw = img.shape[:2]
    budget_w = w * (1 - 2 * padding_per_side)
    budget_h = h * (1 - 2 * padding_per_side)
    scale = min(budget_w / iw, budget_h / ih)
    nw, nh = max(1, round(iw * scale)), max(1, round(ih * scale))
    resized = cv2.resize(img, (nw, nh))
    canvas = np.full((h, w, 3), BG, np.uint8)
    x0, y0 = (w - nw) // 2, (h - nh) // 2
    canvas[y0 : y0 + nh, x0 : x0 + nw] = resized
    return canvas


def hstack_gutter(row_cells):
    gutter = np.full((CELL_H, GUTTER, 3), BG, np.uint8)
    parts = [c for cell in row_cells for c in (cell, gutter)]
    return np.hstack(parts[:-1])


def build_sheets():
    paths = sorted(CROPS.rglob("*.jpg"))
    if not paths:
        print("no crops yet")
        return []

    written = []
    for sheet_no, start in enumerate(range(0, len(paths), PER_SHEET)):
        chunk = paths[start : start + PER_SHEET]
        cells = []
        for i, p in enumerate(chunk):
            img = cv2.imread(str(p))
            cell = np.full((CELL_H, CELL_W, 3), BG, np.uint8) if img is None else fit_letterbox(img, CELL_W, CELL_H)
            label = str(start + i)
            cv2.rectangle(cell, (0, 0), (CELL_W - 1, CELL_H - 1), (90, 90, 90), 1)
            cv2.rectangle(cell, (0, 0), (34, 16), (0, 0, 0), -1)
            cv2.putText(cell, label, (3, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)
            cells.append(cell)

        blank = np.full((CELL_H, CELL_W, 3), BG, np.uint8)
        cells += [blank] * (-len(cells) % COLS)
        rows = [hstack_gutter(cells[r : r + COLS]) for r in range(0, len(cells), COLS)]
        vgutter = np.full((GUTTER, rows[0].shape[1], 3), BG, np.uint8)
        sheet = np.vstack([part for row in rows for part in (row, vgutter)][:-1])

        out = SHEETS / f"sheet_{sheet_no:02d}.png"
        cv2.imwrite(str(out), sheet)
        written.append((out, sheet))
    return written


sheets = build_sheets()
for out, sheet in sheets:
    print(out.relative_to(REPO))
    plt.figure(figsize=(18, 18 * sheet.shape[0] / sheet.shape[1]))
    plt.imshow(cv2.cvtColor(sheet, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(out.name)
    plt.show()

## 6 · Spot-check the crop geometry

Draws three manifest bboxes back onto their source frame. If these boxes hug the
people the way the crops do, the eval set matches what production sends.

In [ ]:
def spot_check(n=3):
    image_by_stem = {p.stem[:24]: p for p in IMAGE_SOURCES}
    picked = [(cid, m) for cid, m in sorted(manifest.items()) if m["source"] in image_by_stem][:n]
    if not picked:
        print("no image-sourced crops to check")
        return

    for crop_id, meta in picked:
        frame = cv2.imread(str(image_by_stem[meta["source"]]))
        x1, y1, x2, y2 = meta["bbox"]
        annotated = frame.copy()
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 3)

        matches = list(CROPS.rglob(f"*__{crop_id}.jpg"))
        crop = cv2.imread(str(matches[0])) if matches else frame[y1:y2, x1:x2]

        fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [3, 1]})
        axes[0].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"{meta['source']} - bbox {meta['bbox']}")
        axes[1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"crop {crop.shape[1]}x{crop.shape[0]}")
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()


spot_check()

## 7 · Summary + how to annotate

In [ ]:
SLUGS = {
    "sleeping": "sleeping",
    "phone": "using phone",
    "computer": "working with computer",
    "talking": "talking with someone",
    "not_focusing": "not_focusing",
    "idle": "idle",
    "skip": "(excluded from the eval set)",
}

counts: dict[str, int] = {}
labelled = 0
for p in CROPS.rglob("*.jpg"):
    label = p.parent.name if p.parent.name in SLUGS else p.stem.split("__", 1)[0]
    counts[label] = counts.get(label, 0) + 1
    if label in SLUGS:
        labelled += 1

total = sum(counts.values())
print(f"{total} crops in {CROPS.relative_to(REPO)}\n")
for label, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {label:<16} {n:>4}")

print(f"\n{labelled}/{total} labelled\n")
print("To annotate, either:")
print("  a) rename the prefix   unlabeled__foo.jpg -> phone__foo.jpg")
print("  b) or move the file    crops/phone/unlabeled__foo.jpg")
print("\nValid slugs:")
for slug, meaning in SLUGS.items():
    print(f"  {slug:<16} {meaning}")
print("\nUse 'skip' for occluded, truncated or genuinely ambiguous crops -")
print("they are excluded from scoring rather than forced into a class.")
print("\nThen run 02_action_prompt_eval.ipynb.")

## 8 · Import the hand-curated "smart office" set

`notebooks/input/smart office/` holds pre-labelled crops named by prefix:
`nfc`/`nonfocus` = not_focusing, `phone` = using phone, `sleeping`/`slp` =
sleeping, `spk` = talking with someone. These fill exactly the gap the first
round left: `using phone` and `talking with someone` had 3 and 2 labelled
examples respectively, and `not_focusing`/`sleeping` had none.

Several `spk` images deliberately keep **both** people in frame - e.g.
`spk-2.png` shows the full conversation, not just one person. Production's
real crop is a single tight per-person box, which likely explains why
`talking with someone` scored 0% in the first sweep. That is a real question
about crop geometry, not just labelling, so this section produces **two**
crop sets instead of silently picking one:

- **`crops_asis/`** - the images as given, no re-detection. Tests the prompt
  under the best-case context you curated; a ceiling if crop geometry is
  ever revisited.
- **`crops/`** (merged into the existing set) - re-run through the same
  YOLO + tight-crop pipeline as every other crop here, so it is directly
  comparable to the first round and to what production actually sends today.

Labels are trusted as given, not re-verified crop by crop - the two QA
sheets at the end are there so anything that looks wrong is easy to catch
by eye rather than silently trusted.

In [ ]:
SMART_OFFICE_DIR = REPO / "notebooks" / "input" / "smart office"
CROPS_ASIS = EVAL / "crops_asis"
CROPS_ASIS.mkdir(parents=True, exist_ok=True)

# Mirrors configs/config.yaml action_recognition.min_crop_* - kept as a literal
# here rather than parsed from YAML, since this is a one-off diagnostic, not
# something that needs to track config drift.
PROD_MIN_HEIGHT, PROD_MIN_WIDTH, PROD_MIN_AREA = 200, 80, 30000


def smart_office_label(path: Path) -> str | None:
    stem = path.stem.lower()
    if stem.startswith(("nfc", "nonfocus")):
        return "not_focusing"
    if stem.startswith("phone"):
        return "using phone"
    if stem.startswith(("sleeping", "slp")):
        return "sleeping"
    if stem.startswith("spk"):
        return "talking with someone"
    return None


SLUG_OF_LABEL = {
    "not_focusing": "not_focusing",
    "using phone": "phone",
    "sleeping": "sleeping",
    "talking with someone": "talking",
}


def dedupe_by_content(paths: list[Path]) -> list[Path]:
    """Drop byte-identical files (e.g. "nfc-1 copy.png") before they get
    counted twice."""
    seen: dict[str, Path] = {}
    out = []
    for p in sorted(paths):
        digest = hashlib.sha1(p.read_bytes()).hexdigest()
        if digest in seen:
            print(f"  ! duplicate of {seen[digest].name}, skipped: {p.name}")
            continue
        seen[digest] = p
        out.append(p)
    return out


so_files = dedupe_by_content(list(SMART_OFFICE_DIR.glob("*.png"))) if SMART_OFFICE_DIR.exists() else []
unrecognised = [p.name for p in so_files if smart_office_label(p) is None]
so_files = [p for p in so_files if smart_office_label(p) is not None]

print(f"{len(so_files)} smart-office images (after dedup, after dropping unrecognised names)")
if unrecognised:
    print(f"  ? unrecognised filename pattern, skipped: {unrecognised}")

In [ ]:
# --- crops_asis: the images exactly as given -----------------------------
asis_manifest_path = EVAL / "manifest_asis.json"
asis_manifest = json.loads(asis_manifest_path.read_text()) if asis_manifest_path.exists() else {}

asis_new = 0
for p in so_files:
    label = smart_office_label(p)
    img = cv2.imread(str(p))
    if img is None:
        print(f"  ! unreadable, skipped: {p.name}")
        continue

    digest = hashlib.sha1(p.read_bytes()).hexdigest()[:8]
    crop_id = f"smartoffice_{p.stem.replace(' ', '_')}_{digest}"
    dest = CROPS_ASIS / f"{SLUG_OF_LABEL[label]}__{crop_id}.jpg"
    if not dest.exists():
        cv2.imwrite(str(dest), img)
        asis_new += 1

    asis_manifest[crop_id] = {
        "source_file": p.name, "truth": label,
        "width": img.shape[1], "height": img.shape[0],
    }

asis_manifest_path.write_text(json.dumps(asis_manifest, indent=2, sort_keys=True))
print(f"crops_asis: {asis_new} new, {len(asis_manifest)} total -> {CROPS_ASIS.relative_to(REPO)}")

In [ ]:
# --- crops/: re-detected, tight single-person crop, merged into the existing
# production-faithful set. Reuses detect_persons / clip_bbox / crop_is_usable
# / bbox_area from section 4 above - same pipeline, same quality gate.
prod_new = 0
multi_person = []          # images where >1 person was detected - the partner
                            # in a "talking" shot may have been cropped out
below_prod_gate = []        # would the NEW production min-crop filter reject this?
below_quality_gate = []     # rejected outright, never written

for p in so_files:
    label = smart_office_label(p)
    img = cv2.imread(str(p))
    if img is None:
        continue

    dets = detect_persons(img)
    if not dets:
        print(f"  ! no person detected, skipped: {p.name}")
        continue
    if len(dets) > 1:
        multi_person.append(p.name)

    top = max(dets, key=bbox_area)  # largest = the deliberately-framed subject
    x1, y1, x2, y2 = clip_bbox(img, top["bbox"])
    if x2 <= x1 or y2 <= y1:
        continue
    crop = img[y1:y2, x1:x2].copy()
    if not crop_is_usable(crop):
        below_quality_gate.append(p.name)
        continue

    ch, cw = crop.shape[:2]
    if ch < PROD_MIN_HEIGHT or cw < PROD_MIN_WIDTH or ch * cw < PROD_MIN_AREA:
        below_prod_gate.append((p.name, cw, ch))

    digest = hashlib.sha1(f"smartoffice|{p.name}|{x1},{y1},{x2},{y2}".encode()).hexdigest()[:8]
    crop_id = f"smartoffice_{p.stem.replace(' ', '_')}_{digest}"
    dest = CROPS / f"{SLUG_OF_LABEL[label]}__{crop_id}.jpg"
    if not dest.exists():
        cv2.imwrite(str(dest), crop)
        prod_new += 1

    manifest[crop_id] = {
        "source": f"smartoffice/{p.name}", "frame_num": 0,
        "bbox": [x1, y1, x2, y2], "confidence": round(top["confidence"], 4),
        "width": cw, "height": ch, "truth": label,
        "detected_person_count": len(dets),
    }

manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"crops/ (production-faithful): {prod_new} new, {len(manifest)} total in manifest\n")

print(f"images with >1 person detected (talking partner may be cropped out): {len(multi_person)}/{len(so_files)}")
for n in multi_person:
    print("   ", n)

print(f"\nbelow the eval quality gate, never written: {len(below_quality_gate)}")
for n in below_quality_gate:
    print("   ", n)

print(f"\nwould be rejected by the production min-crop filter "
      f"(h>={PROD_MIN_HEIGHT}, w>={PROD_MIN_WIDTH}, area>={PROD_MIN_AREA}): {len(below_prod_gate)}")
for n, w, h in below_prod_gate:
    print(f"    {n:24} {w}x{h}")

### QA sheets for the imported crops

Same letterbox + gutter rendering as section 5, tagged with the **trusted
label** instead of an index - the point is to catch a wrong detection or a
mislabelled file by eye, not to re-annotate.

In [ ]:
def build_qa_sheet(paths, out_path, tag_fn):
    if not paths:
        return None
    cells = []
    for p in paths:
        img = cv2.imread(str(p))
        cell = np.full((CELL_H, CELL_W, 3), BG, np.uint8) if img is None else fit_letterbox(img, CELL_W, CELL_H)
        cv2.rectangle(cell, (0, 0), (CELL_W - 1, CELL_H - 1), (90, 90, 90), 1)
        cv2.rectangle(cell, (0, 0), (CELL_W - 1, 16), (0, 0, 0), -1)
        cv2.putText(cell, tag_fn(p)[:20], (3, 12), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (0, 255, 255), 1)
        cells.append(cell)
    cells += [np.full((CELL_H, CELL_W, 3), BG, np.uint8)] * (-len(cells) % COLS)
    rows = [hstack_gutter(cells[r : r + COLS]) for r in range(0, len(cells), COLS)]
    vgutter = np.full((GUTTER, rows[0].shape[1], 3), BG, np.uint8)
    sheet = np.vstack([part for row in rows for part in (row, vgutter)][:-1])
    cv2.imwrite(str(out_path), sheet)
    return sheet


def show_sheet(sheet, out_path, title):
    if sheet is None:
        print(f"({title}: nothing to show)")
        return
    print(out_path.relative_to(REPO))
    plt.figure(figsize=(18, 18 * sheet.shape[0] / sheet.shape[1]))
    plt.imshow(cv2.cvtColor(sheet, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(title)
    plt.show()


asis_paths = sorted(CROPS_ASIS.glob("*.jpg"))
show_sheet(
    build_qa_sheet(asis_paths, SHEETS / "smartoffice_asis.png", lambda p: p.stem.split("__")[0]),
    SHEETS / "smartoffice_asis.png", "smart office - as-is",
)

prod_paths = sorted(p for p in CROPS.glob("*.jpg") if "smartoffice" in p.stem)
show_sheet(
    build_qa_sheet(prod_paths, SHEETS / "smartoffice_prodcrop.png", lambda p: p.stem.split("__")[0]),
    SHEETS / "smartoffice_prodcrop.png", "smart office - production-faithful re-crop",
)